# Profitable Wallet Analysis

Phase 2 (volatility path): load preprocessed trades, compute volatility metrics, select top wallets, and plot PnL charts.

In [1]:
import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_pnl_bars,
    plot_wallet_returns,
    plot_cumulative_pnl_by_wallet,
    plot_combined_cumulative_pnl,
)


In [2]:
# ── configuration ─────────────────────────────────────────────────────────────
TRADES_DIR = Path("../../data/polygon_trades_processed")


In [3]:
# ── load preprocessed trades ──────────────────────────────────────────────────
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
if not trade_files:
    raise FileNotFoundError(f"No parquet files found in {TRADES_DIR}")

df = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)
df["dt"] = pd.to_datetime(df["dt"], utc=True)
df["usdc_amount"]     = df["usdc_amount"].astype(float)
df["final_value_usdc"] = df["final_value_usdc"].astype(float)
df["quantity"]        = df["quantity"].astype(float)
df["position"]        = df["position"].astype(float)
df["price"]           = df["price"].astype(float)

# backward-compatible aliases
df["trade_value_usdc"] = df["usdc_amount"]
df["size"]             = df["quantity"]
df["transactionHash"]  = df["tx_hash"]

# PnL per fill
df["pnl"] = np.where(
    df["side"] == "BUY",
    df["final_value_usdc"] - df["trade_value_usdc"],
    df["trade_value_usdc"] - df["final_value_usdc"],
)
# notional per fill
df["notional"] = np.where(
    df["side"] == "BUY",
    df["trade_value_usdc"],
    df["size"] * (1 - df["price"]),
)

# ── derive train/test boundary from is_train flag ───────────────────────────
END_DATE_TRAIN = df.loc[df["is_train"], "dt"].max().date()

print(f"Loaded {len(df):,} records  wallets={df['wallet'].nunique():,}  "
      f"train={df['is_train'].sum():,}  test={(~df['is_train']).sum():,}")


Loaded 141,129 records  wallets=1,464  train=122,239  test=18,890


In [4]:
# ── compute volatility metrics on training slice ───────────────────────────────
df_train = df[df["is_train"]]
wallet_vol_train, _ = compute_wallet_metrics(df_train)
wallet_vol_train = wallet_vol_train.sort_values("pnl_volatility").reset_index(drop=True)

print(f"Training wallets with volatility metric: {wallet_vol_train['pnl_volatility'].notna().sum():,}")
wallet_vol_train.head(10)


Training wallets with volatility metric: 949


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
0,0x02e79614b74e11f584bb6e2a6ce212772063c57e,0.009633,2.0,2.0,828.400000,1111.600000,1.0,0.500315,1.341864
1,0x51e1df8b3096d21b953c1c48587509eedaa1d2ad,0.021416,2.0,2.0,1108.400000,711.600000,1.0,0.500843,0.642006
2,0x80521327c2759e27fceb76112d1f304ded701124,0.063142,2.0,2.0,406.999999,493.101387,1.0,0.502939,1.211551
3,0x09bfd6b8347e4e24d4913a4a9176d0bebe92da1e,0.115935,2.0,2.0,1804.000000,1096.000000,1.0,0.503650,0.607539
4,0xf756993c74750e9c27657e8b00d6eee452938fe3,0.126372,2.0,2.0,1143.000000,732.000000,1.0,0.504918,0.640420
5,0xdedd4ef099c1bb2dbd110bbb2106ff337186705c,0.138316,2.0,2.0,729.550000,975.450000,1.0,0.504844,1.337057
6,0x38999e0c830e80a4de3b6672756e6ce122f23b50,0.147620,2.0,2.0,1067.100000,687.900000,1.0,0.505887,0.644644
7,0x66348857f4abdb46569747b605e06fbe8e291c75,0.190679,2.0,2.0,722.500000,937.500000,1.0,0.506880,1.297578
8,0xe89eec6251ed51e4aed9f5c07f7fb884093cd1bd,0.203402,2.0,2.0,400.999998,500.433719,1.0,0.509092,1.247964
9,0xdc4d591c23abb7f7da3149d5fa7c6122c4a7af28,0.244182,2.0,1.0,599.999998,565.459154,1.0,1.000000,0.942432


In [5]:
# ── select best wallets based on training-period volatility metrics ─────────────
filtered_wallets = filter_wallets_by_volatility(
    wallet_vol_train,
    min_buckets=20,
    max_top5_pnl_pct=1,#0.4,
    max_top_market_pnl_pct=0.5,
)
print(f"Selected wallets: {len(filtered_wallets)}")
filtered_wallets.head(20)


Selected wallets: 36


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,top5_pnl_pct,top_market_pnl_pct,return
0,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,86.991170,95.0,11.0,3.754735e+05,344182.639640,0.879996,0.413717,0.916663
1,0xb45a797faa52b0fd8adc56d30382022b7b12192c,71.212290,99.0,22.0,3.562734e+05,131838.674837,0.872896,0.443724,0.370049
2,0x6ade597c0e2b43c0bf3542cada8a5e330d73f5b0,44.573612,29.0,14.0,9.184596e+04,58153.718184,0.954313,0.462911,0.633166
3,0xed8fc3b266ddf409b17e0734617a520a3be85aa7,23.367272,29.0,14.0,2.961211e+04,22160.341498,0.847372,0.487655,0.748354
4,0x661daf6af6d884012dd6db73c09d72e8be224dc6,13.185659,193.0,35.0,8.236577e+04,21750.299545,0.950907,0.374416,0.264070
5,0xcd3dde7cf49f0cc08cd3bf329c2334e4dbe4e7eb,8.722294,28.0,15.0,9.077051e+03,8785.428881,0.852429,0.357074,0.967873
6,0x4a867a1f268e153de1d359e337914a714fabd3e0,4.068614,555.0,47.0,9.441027e+04,7260.252650,0.576430,0.258323,0.076901
7,0xd50caddb396e5fa6a34361a87b1763de7d4376c3,3.807166,33.0,4.0,7.875751e+03,5250.960373,0.525775,0.406759,0.666725
8,0x33bcb6e9bd44be709122b2940c55e34f1a7e37dc,7.084659,25.0,10.0,6.984779e+03,4054.254124,0.818298,0.437947,0.580441
9,0xe91171f655be1568e4c63f29663c0028649e3d4e,3.936828,158.0,66.0,1.950694e+04,3180.327346,0.819714,0.270709,0.163036


In [6]:
# ── evaluate on test set ─────────────────────────────────────────────────────
best_wallet_set = set(filtered_wallets["wallet"])
df_test_best = df[~df["is_train"] & df["wallet"].isin(best_wallet_set)]
wallet_vol_test, _ = compute_wallet_metrics(df_test_best)

METRIC_COLS = ["wallet", "pnl_volatility", "num_buckets", "num_markets",
               "total_notional", "total_pnl", "return",
               "top5_pnl_pct", "top_market_pnl_pct"]
comparison = (
    filtered_wallets[METRIC_COLS]
    .merge(wallet_vol_test[METRIC_COLS], on="wallet", how="left", suffixes=("_train", "_test"))
    .sort_values("total_pnl_train", ascending=False)
    .reset_index(drop=True)
)
comparison["wallet_short"] = comparison["wallet"].str[:6] + "..." + comparison["wallet"].str[-4:]
print(f"Wallets with test data: {comparison['total_pnl_test'].notna().sum()} / {len(comparison)}")
comparison


Wallets with test data: 32 / 36


,wallet,pnl_volatility_train,num_buckets_train,num_markets_train,total_notional_train,total_pnl_train,return_train,top5_pnl_pct_train,top_market_pnl_pct_train,pnl_volatility_test,num_buckets_test,num_markets_test,total_notional_test,total_pnl_test,return_test,top5_pnl_pct_test,top_market_pnl_pct_test,wallet_short
0,0x6a72f61820b26b1fe4d956e17b6dc2a1ea3033ee,86.991170,95.0,11.0,3.754735e+05,344182.639640,0.916663,0.879996,0.413717,NaN,21.0,2.0,129784.803802,-121553.923659,-0.936581,NaN,NaN,0x6a72...33ee
1,0xb45a797faa52b0fd8adc56d30382022b7b12192c,71.212290,99.0,22.0,3.562734e+05,131838.674837,0.370049,0.872896,0.443724,NaN,1.0,1.0,10200.000000,-10200.000000,-1.000000,NaN,NaN,0xb45a...192c
2,0x6ade597c0e2b43c0bf3542cada8a5e330d73f5b0,44.573612,29.0,14.0,9.184596e+04,58153.718184,0.633166,0.954313,0.462911,16.160494,8.0,2.0,24658.893707,26088.801892,1.057988,0.998126,0.785777,0x6ade...f5b0
3,0xed8fc3b266ddf409b17e0734617a520a3be85aa7,23.367272,29.0,14.0,2.961211e+04,22160.341498,0.748354,0.847372,0.487655,NaN,3.0,3.0,2580.099997,-2580.099997,-1.000000,NaN,NaN,0xed8f...5aa7
4,0x661daf6af6d884012dd6db73c09d72e8be224dc6,13.185659,193.0,35.0,8.236577e+04,21750.299545,0.264070,0.950907,0.374416,NaN,16.0,4.0,3442.161787,-359.311241,-0.104385,NaN,NaN,0x661d...4dc6
5,0xcd3dde7cf49f0cc08cd3bf329c2334e4dbe4e7eb,8.722294,28.0,15.0,9.077051e+03,8785.428881,0.967873,0.852429,0.357074,0.814155,2.0,1.0,100.312500,212.187500,2.115265,1.000000,1.000000,0xcd3d...e7eb
6,0x4a867a1f268e153de1d359e337914a714fabd3e0,4.068614,555.0,47.0,9.441027e+04,7260.252650,0.076901,0.576430,0.258323,9.861488,79.0,13.0,15812.286724,1162.892705,0.073544,2.271061,1.318262,0x4a86...d3e0
7,0xd50caddb396e5fa6a34361a87b1763de7d4376c3,3.807166,33.0,4.0,7.875751e+03,5250.960373,0.666725,0.525775,0.406759,4.406328,2.0,1.0,759.999987,381.839848,0.502421,1.000000,1.000000,0xd50c...76c3
8,0x33bcb6e9bd44be709122b2940c55e34f1a7e37dc,7.084659,25.0,10.0,6.984779e+03,4054.254124,0.580441,0.818298,0.437947,4.408167,9.0,3.0,1154.538845,288.976575,0.250296,1.002530,0.989353,0x33bc...37dc
9,0xe91171f655be1568e4c63f29663c0028649e3d4e,3.936828,158.0,66.0,1.950694e+04,3180.327346,0.163036,0.819714,0.270709,30.738283,40.0,20.0,3979.864399,2623.205601,0.659119,1.259383,0.997561,0xe911...3d4e


In [7]:
# ── plots ─────────────────────────────────────────────────────────────────────
plot_wallet_pnl_bars(comparison).show(renderer="browser")
plot_wallet_returns(comparison).show(renderer="browser")


In [8]:
# ── per-wallet and combined cumulative PnL ────────────────────────────────────
df_best = df[df["wallet"].isin(best_wallet_set)].copy()
df_best["dt_floored"] = df_best["dt"].dt.floor("1h")
buckets_full = (
    df_best.groupby(["wallet", "dt_floored", "condition_id"], sort=False)
    .agg(notional=("notional", "sum"), pnl=("pnl", "sum"))
    .reset_index()
)
buckets_full = buckets_full[buckets_full["notional"] > 0].copy()

split_line = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
top_wallets_plot = filtered_wallets.head(20)["wallet"].tolist()

plot_cumulative_pnl_by_wallet(
    buckets_full, top_wallets_plot, split_date=split_line
).show(renderer="browser")

plot_combined_cumulative_pnl(
    buckets_full, best_wallet_set, split_date=split_line
).show(renderer="browser")
